# Assignment 1

Deadline: 19.03.2026, 12:00 CET

Matilde Venturi , 25-741-059 , matilde.venturi@uzh.ch

In [8]:
# Import standard libraries
import os
import sys
import time # To compute runtimes
from typing import Optional

# Import third-party libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import local modules
project_root = os.path.dirname(os.path.dirname(os.getcwd()))   # Change this path if needed
src_path = os.path.join(project_root, 'qpmwp-course', 'src') #changed for Macbook
sys.path.append(project_root)
sys.path.append(src_path)
from estimation.covariance import Covariance, is_pos_def, make_pos_def
from estimation.expected_return import ExpectedReturn
from optimization.constraints import Constraints
from optimization.optimization import Optimization, Objective
from optimization.optimization_data import OptimizationData
from optimization.quadratic_program import QuadraticProgram, USABLE_SOLVERS
from helper_functions import simulate_correlated_gbm

## 1. Solver Horse Race

### 1.a)
(3 points)

Generate a synthetic dataset of dimension TxN, T=1000, N=50, and compute a vector of expected returns, q, and a covariance matrix, P, using classes ExpectedReturn and Covariance respectively.

In [9]:

# Set the dimensions
T = 1000       # Number of time steps
N = 50         # Number of assets
rnd_seed = 42  # Random seed for reproducibility

# Set random seed for reproducibility
np.random.seed(rnd_seed)


# Generate a random mean vector and covariance matrix.
data = np.random.randn(T, N)   
df = pd.DataFrame(data)
# Make sure the covariance matrix is positive definite.

scalefactor = 1  # could be set to 252 (trading days) for annualized returns

mu=np.mean(data,axis=0)
sigma = np.cov(data, rowvar=False)


# Generate correlated geometric Brownian motion paths and compute discrete returns
prices = simulate_correlated_gbm(mu=mu, sigma=sigma, T=T, random_seed=None)
returns = prices.pct_change().dropna()

expected_return = ExpectedReturn(
    method='geometric',
    scalefactor=scalefactor
)

# Compute the vector of expected returns from df using class ExpectedReturn
q = expected_return.estimate(X=df, inplace=False).to_numpy()

# Compute the covariance matrix from df using class Covariance
covariance = Covariance(method="pearson")
P = covariance.estimate(X=df, inplace=False)


# Display the results
print("Vector of expected returns (q):")
print(q)

print("\nCovariance matrix (P):")
print(P)

Vector of expected returns (q):
[-0.02616732 -0.02845047 -0.07213305 -0.03652317 -0.10827774 -0.00025891
 -0.04557468 -0.01036338 -0.00372751 -0.0341873  -0.07308284 -0.02774128
 -0.05082311 -0.04618358  0.00240935 -0.02947732  0.00938743 -0.10037764
 -0.0534574  -0.0174171  -0.00270007 -0.00689962 -0.01820603 -0.05462584
 -0.04130829 -0.00075405 -0.04404641 -0.05287391  0.01977684 -0.07626421
 -0.04203406 -0.08873625 -0.03307816  0.00179157 -0.03358332 -0.05779126
 -0.0255365  -0.04088994 -0.07637818  0.01807297 -0.00586807 -0.0334113
 -0.08268077 -0.03623672 -0.0568171  -0.01050744 -0.0426385  -0.07396444
 -0.03350186 -0.03635502]

Covariance matrix (P):
          0         1         2         3         4         5         6   \
0   1.032911  0.017379 -0.002861  0.015165 -0.035948  0.018677  0.041287   
1   0.017379  0.921750 -0.020559  0.054517  0.007334 -0.020952  0.032308   
2  -0.002861 -0.020559  1.025287  0.043178 -0.000237 -0.007590 -0.017945   
3   0.015165  0.054517  0.04317

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


### 1.b)
(3 points)

Instantiate a constraints object by injecting column names of the return series created in 1.a) as ids and add:
- a budget constaint (i.e., asset weights have to sum to one)
- lower bounds of 0.0 for all assets
- upper bounds of 0.2 for all assets
- group contraints such that the sum of the weights of the first 15 assets is <= 0.3, the sum of assets 16 to 45 is <= 0.4 and the sum of assets 41 to 50 is <= 0.5

In [10]:
# Instantiate the Constraints class
constraints = Constraints(ids = returns.columns.tolist())

# Add budget constraint
constraints.add_budget(rhs=1,sense="=")

# Add box constraints (i.e., lower and upper bounds)
constraints.add_box(lower=0,upper=0.2)

# Add linear constraints
G=pd.DataFrame(np.zeros((3,N)),columns=constraints.ids)
G.iloc[0,0:15]=1
G.iloc[1,15:45]=1
G.iloc[2,40:50]=1
h=pd.Series([0.3,0.4,0.5])
constraints.add_linear(G=G,sense="<=",rhs=h)

# Display some columns of the G matrix to verify that the constraints have been set up correctly
constraints.linear['G'][['Asset_1', 'Asset_15', 'Asset_16', 'Asset_40', 'Asset_41', 'Asset_50']]

,Asset_1,Asset_15,Asset_16,Asset_40,Asset_41,Asset_50
0,1.0,1.0,0.0,0.0,0.0,0.0
1,0.0,0.0,1.0,1.0,1.0,0.0
2,0.0,0.0,0.0,0.0,1.0,1.0


### 1.c) 
(4 points)

Solve a Mean-Variance optimization problem (using coefficients P and q in the objective function) which satisfies the above defined constraints.
Repeat the task for all open-source solvers in qpsolvers that you could install and compare the results in terms of:

- runtime
- accuracy: value of the primal problem.
- reliability: are all constraints fulfilled? Extract primal residuals, dual residuals and duality gap.

Generate a DataFrame with the solvers as column names and the following row index: 'solution_found': bool, 'objective': float, 'primal_residual': float, 'dual_residual': float, 'duality_gap': float, 'runtime': float.

Put NA's for solvers where the optimization failed for some reason.




In [ ]:
# Extract the constraints in the format required by the solver
GhAb = constraints.to_GhAb()

#I could do this but then it crashes
#import qpsolvers
#available_solvers = qpsolvers.available_solvers

# Define a dictionary to store the results in case a solver fails
result_on_fail =  {
    'solution_found': False,
    'objective_builtin': np.nan,
    'primal_residual': np.nan,
    'dual_residual' : np.nan,
    'duality_gap': np.nan,
    'runtime': np.nan,
}

# Loop over solvers, instantiate the quadratic program, solve it and store the results in a dictionary.
risk_aversion=1 #?


#idea: key del dizionario il tipo di solvers, value=lista di cose da controllare?

available_solvers = ['osqp', 'cvxopt']
dict={}

for s in available_solvers:

    qp = QuadraticProgram(
    P = P.to_numpy() * risk_aversion,
    q = q * -1,
    G = GhAb['G'],
    h = GhAb['h'],
    A = GhAb['A'],
    b = GhAb['b'],
    lb = constraints.box['lower'].to_numpy(),
    ub = constraints.box['upper'].to_numpy(),
    solver = s, #itero nei solvers
    )
   

    t0 = time.perf_counter()
    qp.solve()
    runtime = time.perf_counter() - t0
    qp.objective_value()
    solution = qp.results.get('solution')
    solution
    
   
    
    
    if solution is None:
        dict[s]=result_on_fail
    else:
        compare={
            "solution found":solution.found,
             "objective":qp.objective_value(),
             "primal residual":solution.primal_residual(),
             "dual_residual":solution.dual_residual(),
             "duality gap":solution.duality_gap()[0],
             "runtime":runtime}
        dict[s]=compare
        




datafr=pd.DataFrame(dict)

Print and visualize the results

In [12]:
print(datafr)

                     osqp    cvxopt
solution found       True      True
objective        0.036745  0.036826
primal residual   0.00025       0.0
dual_residual    0.000001       0.0
duality gap       0.00008       0.0
runtime          0.002158  0.000897


## 2. Analytical Solution to Minimum-Variance Problem

(5 points)

- Create a `MinVariance` class that follows the structure of the `MeanVariance` class.
- Implement the `solve` method in `MinVariance` such that if `solver_name = 'analytical'`, the analytical solution is computed and stored within the object (if such a solution exists). If not, call the `solve` method from the parent class.
- Create a `Constraints` object by injecting the same ids as in part 1.b) and add a budget constraint.
- Instantiate a `MinVariance` object by setting `solver_name = 'analytical'` and passing instances of `Constraints` and `Covariance` as arguments.
- Create an `OptimizationData` object that contains an element `return_series`, which consists of the synthetic data generated in part 1.a).
- Solve the optimization problem using the created `MinVariance` object and compare the results to those obtained in part 1.c).


In [ ]:
# Define class MinVariance
class MinVariance(Optimization):

    def __init__(self,
                 constraints: Constraints,
                 covariance: Optional[Covariance] = None,
                 **kwargs):
        super().__init__(
            constraints=constraints,
            **kwargs
        )
        self.covariance = Covariance() if covariance is None else covariance

    def set_objective(self, optimization_data: OptimizationData) -> None: #should this contain the Sigma=?
        X = optimization_data['return_series']
        covmat = self.covariance.estimate(X=X, inplace=False)
        mu = np.zeros(X.shape[1])
        self.objective = Objective(
            q = mu ,
            P = covmat * 2,
        )
        return None

    def solve(self) -> None:

     if self.params.get("solver_name") == "analytical":
        # Get the coefficients of the objective function
        obj_coeff = self.objective.coefficients
        if 'P' not in obj_coeff.keys() or 'q' not in obj_coeff.keys():
            raise ValueError("Objective must contain 'P' and 'q'.")

        # Ensure that P and q are numpy arrays
        obj_coeff['P'] = to_numpy(obj_coeff['P'])
        obj_coeff['q'] = to_numpy(obj_coeff['q'])

        P=obj_coeff['P']
        constraints = self.constraints
        GhAb = constraints.to_GhAb()
        A = constraints.linear['G'].to_numpy() if constraints.linear['linear_type'] != 'NA' else None
        b = constraints.linear['rhs'].to_numpy() if constraints.linear['linear_type'] != 'NA' else None

        x= ((np.linalg.inv(P) @ A.T) @ np.linalg.inv(A @ np.linalg.inv(P) @ A.T) @ b )
        self.results['weights']=x
        return 0.5 * (x.T @ P @ x)
        



     else:
        return super().solve()
      


# Create a constraints object with just a budget constraint

constraints = Constraints(ids = returns.columns.tolist())
constraints.add_budget(rhs=1,sense="=")

covariance = Covariance(method='pearson')
covariance.estimate(X=df, inplace=True)

# Instantiate the MinVariance class
minvar = MinVariance(
    constraints=constraints,
    covariance=covariance,
    solver_name="analytical",   
)

# Prepare the optimization data and prepare the optimization problem
optimization_data = OptimizationData(sigma=covariance.matrix)

# Solve the optimization problem and print the weights
minvar.set_objective(optimization_data=optimization_data)
minvar.objective.coefficients

w = minvar.solve()
print(w)



[0.02142052 0.0188707  0.01834376 0.0228983  0.01953711 0.01866364
 0.01996219 0.0187973  0.01330982 0.02079689 0.01537029 0.01331429
 0.01733822 0.01811921 0.01916964 0.02645163 0.02327608 0.01695449
 0.01848531 0.01248319 0.01239853 0.02420393 0.01969428 0.02491885
 0.01749408 0.02261285 0.01993371 0.01556074 0.01910905 0.02010344
 0.02482209 0.02112054 0.02006973 0.02249463 0.0245598  0.01587469
 0.02635297 0.02146324 0.0190494  0.02549138 0.00985113 0.02158351
 0.02240373 0.02442253 0.02216229 0.02253823 0.01656051 0.02854375
 0.01963165 0.02141212]
